# revert, reflog, worktree

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.


## 준비

정상 파일과 두 개의 커밋을 만듭니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 복구 실습용 저장소를 준비합니다.
repo = root / "recovery"
repo.mkdir()
git("init", "-b", "main", cwd=repo)
configure(repo)
(repo / "config.txt").write_text("discount=10\n", encoding="utf-8")
git("add", "config.txt", cwd=repo)
git("commit", "-m", "feat: 할인율 10 설정", cwd=repo)
good = git("rev-parse", "HEAD", cwd=repo)
print(git("log", "--oneline", "-1", cwd=repo))

1fc3d0c feat: 할인율 10 설정


## 기본 실행

잘못된 공개 변경은 revert로 반대 변경을 새 커밋에 기록합니다.

In [3]:
# revert는 할인율을 되돌리는 새 커밋을 만들어 기존 이력을 유지합니다.
(repo / "config.txt").write_text("discount=90\n", encoding="utf-8")
git("add", "config.txt", cwd=repo)
git("commit", "-m", "fix: 할인율 90 설정", cwd=repo)
bad = git("rev-parse", "HEAD", cwd=repo)
git("revert", "--no-edit", bad, cwd=repo)
print((repo / "config.txt").read_text(encoding="utf-8").strip())
print(git("log", "--oneline", "-3", cwd=repo))

discount=10
7d2da3c Revert "fix: 할인율 90 설정"
adba365 fix: 할인율 90 설정
1fc3d0c feat: 할인율 10 설정


## worktree로 두 작업 폴더 사용

별도 브랜치를 별도 폴더에 연결합니다.

In [4]:
# 별도 worktree에서 긴급 수정 브랜치를 사용하고 기본 폴더와 비교합니다.
worktree = root / "urgent-fix"
git("worktree", "add", "-b", "fix/urgent", worktree.as_posix(), cwd=repo)
(worktree / "hotfix.txt").write_text("urgent=true\n", encoding="utf-8")
git("add", "hotfix.txt", cwd=worktree)
git("commit", "-m", "fix: 긴급 설정 추가", cwd=worktree)
print("기본 폴더 브랜치:", git("branch", "--show-current", cwd=repo))
print("추가 폴더 브랜치:", git("branch", "--show-current", cwd=worktree))
print(git("worktree", "list", "--porcelain", cwd=repo))

기본 폴더 브랜치: main
추가 폴더 브랜치: fix/urgent
worktree /private/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-dkgjjpvs/recovery
HEAD 7d2da3ccc129871d1acee1c522a7534505d8d13f
branch refs/heads/main

worktree /private/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-dkgjjpvs/urgent-fix
HEAD 91107ee84ed5795309f88d484b2262536c2c5259
branch refs/heads/fix/urgent


## 실패 재현

아직 공유하지 않은 커밋을 hard reset으로 지우면 일반 log에서 보이지 않습니다.

In [5]:
# 임시 저장소의 미공유 커밋만 이동시켜 일반 log에서 사라짐을 관찰합니다.
git("switch", "-c", "experiment", cwd=repo)
(repo / "idea.txt").write_text("recover me\n", encoding="utf-8")
git("add", "idea.txt", cwd=repo)
git("commit", "-m", "experiment: 복구할 아이디어", cwd=repo)
lost = git("rev-parse", "HEAD", cwd=repo)
git("reset", "--hard", "HEAD~1", cwd=repo)
visible = git("log", "--format=%H", "-3", cwd=repo)
print("일반 log에 있음:", lost in visible)
print("파일 존재:", (repo / "idea.txt").exists())
assert lost not in visible

일반 log에 있음: False
파일 존재: False


## 수정

reflog에서 이동 전 커밋을 찾아 새 브랜치 이름을 붙입니다.

In [6]:
# reflog의 커밋 메시지로 복구 대상을 찾아 새 브랜치 이름을 붙입니다.
reflog = git("reflog", "--format=%H%x09%gs", "-5", cwd=repo)
print(reflog)
assert lost in reflog
candidate = next(
    line.split("\t", 1)[0] for line in reflog.splitlines()
    if "commit: experiment: 복구할 아이디어" in line
)
git("branch", "recovered-idea", candidate, cwd=repo)
recovered = git("show", "recovered-idea:idea.txt", cwd=repo)
print("복구 내용:", recovered)
print("복구 브랜치:", git("rev-parse", "--short", "recovered-idea", cwd=repo))
assert candidate == lost
assert recovered == "recover me"

7d2da3ccc129871d1acee1c522a7534505d8d13f	reset: moving to HEAD~1
608e5fa01db5d6c98fc5833bec1e3595b719e152	commit: experiment: 복구할 아이디어
7d2da3ccc129871d1acee1c522a7534505d8d13f	checkout: moving from main to experiment
7d2da3ccc129871d1acee1c522a7534505d8d13f	revert: Revert "fix: 할인율 90 설정"
adba365ed9bb7978992c09017e89f67a0b19e940	commit: fix: 할인율 90 설정
복구 내용: recover me
복구 브랜치: 608e5fa
